# **2. Preprocessing**

## **2.1 Import libraries**

In [1]:
import numpy as np
import pandas as pd
import xarray as xr
from pathlib import Path

## **2.2 Load raw ERA5 data**

In [ ]:
raw_path = Path("../data/raw/era5_debilt_2010_2025_basic.nc")
processed_dir = Path("../data/processed")
processed_dir.mkdir(parents=True, exist_ok=True)

# load NetCDF, tabular for downstreaming
ds = xr.open_dataset(raw_path, engine="netcdf4")
df = ds.to_dataframe().reset_index()
df = df.sort_values("valid_time").reset_index(drop=True)

In [3]:
# only complete years for clean temporal analysis and modeling
df = df[df["valid_time"] < "2025-01-01"].reset_index(drop=True)
print(df.shape)
print("Start:", df["valid_time"].min())
print("End:", df["valid_time"].max())
df.head()

(131496, 11)
Start: 2010-01-01 00:00:00
End: 2024-12-31 23:00:00


,valid_time,u10,v10,d2m,t2m,msl,sp,tcc,tp,latitude,longitude
0,2010-01-01 00:00:00,-3.206360,-4.415176,270.572205,273.083435,99796.5000,99723.859375,1.000000,0.000034,52.0,5.25
1,2010-01-01 01:00:00,-3.581848,-4.482193,270.281982,273.042145,99785.6250,99713.945312,0.997070,0.000022,52.0,5.25
2,2010-01-01 02:00:00,-3.827850,-4.368881,270.057373,273.026550,99773.6875,99701.164062,0.953491,0.000012,52.0,5.25
3,2010-01-01 03:00:00,-3.738113,-4.246582,269.936249,273.034119,99770.0625,99697.851562,0.975220,0.000010,52.0,5.25
4,2010-01-01 04:00:00,-3.380920,-4.337769,269.910706,273.063354,99808.8125,99736.203125,0.928223,0.000006,52.0,5.25


## **2.4 Convert physical units**

ERA5 uses scientific variables that are correct but for interpretation less intuitive. This means converted columns will be used for modeling but is also better for XAI interpretation. 

In [4]:
# asked AI to convert this correctly

df["tp_mm"] = df["tp"] * 1000
df["t2m_c"] = df["t2m"] - 273.15
df["d2m_c"] = df["d2m"] - 273.15
df["msl_hpa"] = df["msl"] / 100
df["sp_hpa"] = df["sp"] / 100
df["wind_speed_10m"] = np.sqrt(df["u10"] ** 2 + df["v10"] ** 2)

converted_columns = [
    "valid_time",
    "tp_mm",
    "t2m_c",
    "d2m_c",
    "msl_hpa",
    "sp_hpa",
    "tcc",
    "u10",
    "v10",
    "wind_speed_10m",
]

df[converted_columns].describe()

,valid_time,tp_mm,t2m_c,d2m_c,msl_hpa,sp_hpa,tcc,u10,v10,wind_speed_10m
count,131496,131496.000000,131496.000000,131496.000000,131496.000000,131496.000000,131496.000000,131496.000000,131496.000000,131496.000000
mean,2017-07-02 11:30:00,0.100957,10.957495,7.432219,1015.277954,1014.574219,0.678154,0.990631,0.871004,3.946900
min,2010-01-01 00:00:00,0.000000,-14.540375,-17.630005,969.076904,968.383728,0.000000,-9.888504,-9.366302,0.024887
25%,2013-10-01 17:45:00,0.000000,6.100449,3.403435,1009.423096,1008.724014,0.390099,-1.277946,-1.372677,2.498639
50%,2017-07-02 11:30:00,0.000477,10.749496,7.636765,1015.899048,1015.199646,0.838318,1.141426,0.926041,3.658477
75%,2021-04-02 05:15:00,0.039220,15.759033,11.854050,1021.875610,1021.168930,0.997070,3.032928,2.937954,5.051978
max,2024-12-31 23:00:00,12.814045,38.530426,22.034393,1048.081299,1047.350586,1.000000,15.689407,13.243332,16.301874
std,NaN,0.321857,6.753629,5.686004,10.017866,10.010503,0.353004,3.007205,2.917962,1.928640


## **2.5 Create temporal features**

In [5]:
df["year"] = df["valid_time"].dt.year
df["month"] = df["valid_time"].dt.month
df["dayofyear"] = df["valid_time"].dt.dayofyear
df["hour"] = df["valid_time"].dt.hour

# month and hour are circular time variables, not ordinary linear numbers -> therefore defining this on a circle with cos and sin is actually useful. For example January (month 1) and December (month 12) look numerically very far away from each other,
# using cosine and sinus this is resolved

df["month_sin"] = np.sin(2 * np.pi * df["month"] / 12)
df["month_cos"] = np.cos(2 * np.pi * df["month"] / 12)
df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

season_map = {
    12: "winter",
    1: "winter",
    2: "winter",
    3: "spring",
    4: "spring",
    5: "spring",
    6: "summer",
    7: "summer",
    8: "summer",
    9: "autumn",
    10: "autumn",
    11: "autumn",
}
df["season"] = df["month"].map(season_map)

df[
    [
        "valid_time",
        "year",
        "month",
        "season",
        "dayofyear",
        "hour",
        "month_sin",
        "month_cos",
        "hour_sin",
        "hour_cos",
    ]
].head()

,valid_time,year,month,season,dayofyear,hour,month_sin,month_cos,hour_sin,hour_cos
0,2010-01-01 00:00:00,2010,1,winter,1,0,0.5,0.866025,0.000000,1.000000
1,2010-01-01 01:00:00,2010,1,winter,1,1,0.5,0.866025,0.258819,0.965926
2,2010-01-01 02:00:00,2010,1,winter,1,2,0.5,0.866025,0.500000,0.866025
3,2010-01-01 03:00:00,2010,1,winter,1,3,0.5,0.866025,0.707107,0.707107
4,2010-01-01 04:00:00,2010,1,winter,1,4,0.5,0.866025,0.866025,0.500000


## **2.6 Define the target variable**

After some consideration, I will make two options. A 97.5th percentile and a 99th percentile 

In [6]:
# 99th percentile will be used as the main target, will keep 97.5th percentile too in case the target with 99th percentile becomes too sensitive.
# this way we can test more and see if SHAP and LIME remain stable when "extreme precipitation" is less strict.

threshold_975 = df["tp_mm"].quantile(0.975)
threshold_99 = df["tp_mm"].quantile(0.99)

df["extreme_precip_975"] = (df["tp_mm"] >= threshold_975).astype(int)
df["extreme_precip_99"] = (df["tp_mm"] >= threshold_99).astype(int)

print(f"97.5th percentile threshold: {threshold_975:.4f} mm/hour")
print(f"99th percentile threshold: {threshold_99:.4f} mm/hour")

target_summary = pd.DataFrame(
    {
        "count": [df["extreme_precip_975"].sum(), df["extreme_precip_99"].sum()],
        "percentage": [
            df["extreme_precip_975"].mean() * 100,
            df["extreme_precip_99"].mean() * 100,
        ],
    },
    index=["extreme_precip_975", "extreme_precip_99"],
)

target_summary

97.5th percentile threshold: 0.9952 mm/hour
99th percentile threshold: 1.5851 mm/hour


,count,percentage
extreme_precip_975,3292,2.503498
extreme_precip_99,1315,1.000030


## **2.7 Create lagged weather features**
Lagged features give the model recent weather history.

In [8]:
lag_columns = [
    "t2m_c",
    "d2m_c",
    "msl_hpa",
    "sp_hpa",
    "tcc",
    "u10",
    "v10",
    "wind_speed_10m",
    "tp_mm",
]
lag_hours = [1, 3, 6, 12, 24]

for column in lag_columns:
    for lag in lag_hours:
        df[f"{column}_lag{lag}"] = df[column].shift(lag)

lagged_columns = [col for col in df.columns if "_lag" in col]
df[["valid_time"] + lagged_columns[:10]].head(8)

,valid_time,t2m_c_lag1,t2m_c_lag3,t2m_c_lag6,t2m_c_lag12,t2m_c_lag24,d2m_c_lag1,d2m_c_lag3,d2m_c_lag6,d2m_c_lag12,d2m_c_lag24
0,2010-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-01-01 01:00:00,-0.066559,NaN,NaN,NaN,NaN,-2.577789,NaN,NaN,NaN,NaN
2,2010-01-01 02:00:00,-0.107849,NaN,NaN,NaN,NaN,-2.868011,NaN,NaN,NaN,NaN
3,2010-01-01 03:00:00,-0.123444,-0.066559,NaN,NaN,NaN,-3.092621,-2.577789,NaN,NaN,NaN
4,2010-01-01 04:00:00,-0.115875,-0.107849,NaN,NaN,NaN,-3.213745,-2.868011,NaN,NaN,NaN
5,2010-01-01 05:00:00,-0.086639,-0.123444,NaN,NaN,NaN,-3.239288,-3.092621,NaN,NaN,NaN
6,2010-01-01 06:00:00,-0.353394,-0.115875,-0.066559,NaN,NaN,-3.218384,-3.213745,-2.577789,NaN,NaN
7,2010-01-01 07:00:00,-0.509064,-0.086639,-0.107849,NaN,NaN,-3.192047,-3.239288,-2.868011,NaN,NaN


## **2.8 Create rolling weather features**

In [7]:
rolling_windows = [3, 6, 12, 24]
rolling_mean_columns = ["t2m_c", "d2m_c", "msl_hpa", "sp_hpa", "tcc", "wind_speed_10m"]

for column in rolling_mean_columns:
    shifted = df[column].shift(1)
    for window in rolling_windows:
        df[f"{column}_roll{window}_mean"] = shifted.rolling(window=window).mean()

for window in rolling_windows:
    df[f"wind_speed_10m_roll{window}_max"] = (
        df["wind_speed_10m"].shift(1).rolling(window=window).max()
    )
    df[f"tp_mm_roll{window}_sum"] = df["tp_mm"].shift(1).rolling(window=window).sum()

rolling_columns = [col for col in df.columns if "_roll" in col]
df[["valid_time"] + rolling_columns[:10]].head(30)

,valid_time,t2m_c_roll3_mean,t2m_c_roll6_mean,t2m_c_roll12_mean,t2m_c_roll24_mean,d2m_c_roll3_mean,d2m_c_roll6_mean,d2m_c_roll12_mean,d2m_c_roll24_mean,msl_hpa_roll3_mean,msl_hpa_roll6_mean
0,2010-01-01 00:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-01-01 01:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2010-01-01 02:00:00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010-01-01 03:00:00,-0.099284,NaN,NaN,NaN,-2.846141,NaN,NaN,NaN,997.852722,NaN
4,2010-01-01 04:00:00,-0.115723,NaN,NaN,NaN,-3.058126,NaN,NaN,NaN,997.764587,NaN
5,2010-01-01 05:00:00,-0.108653,NaN,NaN,NaN,-3.181885,NaN,NaN,NaN,997.841878,NaN
6,2010-01-01 06:00:00,-0.185303,-0.142293,NaN,NaN,-3.223806,-3.034973,NaN,NaN,998.045207,997.948964
7,2010-01-01 07:00:00,-0.316366,-0.216044,NaN,NaN,-3.216573,-3.137349,NaN,NaN,998.352702,998.058645
8,2010-01-01 08:00:00,-0.521708,-0.315180,NaN,NaN,-3.203603,-3.192744,NaN,NaN,998.624166,998.233022
9,2010-01-01 09:00:00,-0.744822,-0.465062,NaN,NaN,-3.216715,-3.220261,NaN,NaN,998.973328,998.509267


## **2.9 Select modeling columns**

In [9]:
base_features = [
    "t2m_c",
    "d2m_c",
    "msl_hpa",
    "sp_hpa",
    "tcc",
    "u10",
    "v10",
    "wind_speed_10m",
    "month",
    "dayofyear",
    "hour",
    "month_sin",
    "month_cos",
    "hour_sin",
    "hour_cos",
]

feature_columns = base_features + lagged_columns + rolling_columns
target_column = "extreme_precip_99"

modeling_columns = [
    "valid_time",
    "latitude",
    "longitude",
    "tp_mm",
    "season",
    "extreme_precip_975",
    target_column,
] + feature_columns
model_df = df[modeling_columns].copy()

print("Number of features:", len(feature_columns))
print("rows before dropping lag/rolling NA values:", len(model_df))
model_df.head()

Number of features: 92
rows before dropping lag/rolling NA values: 131496


,valid_time,latitude,longitude,tp_mm,season,extreme_precip_975,extreme_precip_99,t2m_c,d2m_c,msl_hpa,...,wind_speed_10m_roll12_mean,wind_speed_10m_roll24_mean,wind_speed_10m_roll3_max,tp_mm_roll3_sum,wind_speed_10m_roll6_max,tp_mm_roll6_sum,wind_speed_10m_roll12_max,tp_mm_roll12_sum,wind_speed_10m_roll24_max,tp_mm_roll24_sum
0,2010-01-01 00:00:00,52.0,5.25,0.033855,winter,0,0,-0.066559,-2.577789,997.965027,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2010-01-01 01:00:00,52.0,5.25,0.021935,winter,0,0,-0.107849,-2.868011,997.856262,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,2010-01-01 02:00:00,52.0,5.25,0.011921,winter,0,0,-0.123444,-3.092621,997.736877,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2010-01-01 03:00:00,52.0,5.25,0.009537,winter,0,0,-0.115875,-3.213745,997.700623,...,NaN,NaN,5.808577,0.067711,NaN,NaN,NaN,NaN,NaN,NaN
4,2010-01-01 04:00:00,52.0,5.25,0.006199,winter,0,0,-0.086639,-3.239288,998.088135,...,NaN,NaN,5.808577,0.043392,NaN,NaN,NaN,NaN,NaN,NaN


## **2.10 Drop rows created incomplete by lagging**

In [10]:
model_df = model_df.dropna().reset_index(drop=True)

print("Rows after dropping lag/rolling NA values:", len(model_df))
print("Remaining missing values:", model_df.isna().sum().sum())
model_df.head()

Rows after dropping lag/rolling NA values: 131472
Remaining missing values: 0


,valid_time,latitude,longitude,tp_mm,season,extreme_precip_975,extreme_precip_99,t2m_c,d2m_c,msl_hpa,...,wind_speed_10m_roll12_mean,wind_speed_10m_roll24_mean,wind_speed_10m_roll3_max,tp_mm_roll3_sum,wind_speed_10m_roll6_max,tp_mm_roll6_sum,wind_speed_10m_roll12_max,tp_mm_roll12_sum,wind_speed_10m_roll24_max,tp_mm_roll24_sum
0,2010-01-02 00:00:00,52.0,5.25,0.000000,winter,0,0,-5.003967,-5.981781,1009.743103,...,2.813718,4.116206,1.938624,0.000000,2.343124,0.000477,5.173072,0.000477,5.979106,0.111580
1,2010-01-02 01:00:00,52.0,5.25,0.000954,winter,0,0,-4.914703,-5.797455,1010.239380,...,2.418847,3.906956,1.771196,0.000000,2.272472,0.000477,4.789077,0.000477,5.979106,0.077724
2,2010-01-02 02:00:00,52.0,5.25,0.005245,winter,0,0,-4.222198,-5.128265,1010.916870,...,2.136089,3.726057,1.395978,0.000954,2.272472,0.000954,4.168169,0.001431,5.979106,0.056744
3,2010-01-02 03:00:00,52.0,5.25,0.010490,winter,0,0,-3.469421,-4.561188,1011.289978,...,1.956567,3.567945,2.013902,0.006199,2.013902,0.006199,3.158924,0.006676,5.979106,0.050068
4,2010-01-02 04:00:00,52.0,5.25,0.009537,winter,0,0,-2.938324,-4.043396,1011.603149,...,1.850372,3.410742,2.013902,0.016689,2.013902,0.016689,2.668931,0.017166,5.979106,0.051022


## **2.11 Chronological train/test split**\
this is time-series data, the split should be chronological instead of random. This reduces temporal leakage 

In [11]:
split_index = int(len(model_df) * 0.8)
split_time = model_df.loc[split_index, "valid_time"]

model_df["split"] = np.where(model_df.index < split_index, "train", "test")

split_summary = model_df.groupby("split")[target_column].agg(["count", "sum", "mean"])
split_summary["extreme_percentage"] = split_summary["mean"] * 100

print("So split time:", split_time)
split_summary

So split time: 2022-01-01 09:00:00


,count,sum,mean,extreme_percentage
split,,,,
test,26295,349,0.013272,1.327249
train,105177,966,0.009185,0.918452


## **2.12 Save processed datasets**

In [12]:
processed_path = processed_dir / "era5_supervised_preprocessed.csv"
feature_list_path = processed_dir / "feature_columns.txt"

model_df.to_csv(processed_path, index=False)
feature_list_path.write_text("\n".join(feature_columns), encoding="utf-8")

print("Saved processed dataset to:", processed_path)
print("Saved feature list to:", feature_list_path)
print("Final shape:", model_df.shape)

Saved processed dataset to: ..\data\processed\era5_supervised_preprocessed.csv
Saved feature list to: ..\data\processed\feature_columns.txt
Final shape: (131472, 100)
